# Cuentos — Opción B: decoder-only (Qwen3-4B) + LoRA · Colab Pro+

Fine-tuning de un modelo **decoder-only en español** para generar cuentos infantiles, condicionado por
**temática** (y opcionalmente título). Arquitectura más natural para texto largo libre → cuentos más
coherentes que mT5.

- Modelo base: **`Qwen/Qwen3-4B-Instruct-2507`** (Apache-2, multilingüe, sin modo *thinking*, no requiere token).
  Para más calidad: `meta-llama/Llama-3.1-8B-Instruct` con QLoRA (requiere token de HF; ver config).
- **LoRA** (PEFT) en bf16 — entra holgado en A100.
- Formato **chat**: system + user(tema/título) → assistant(cuento). La pérdida se calcula **solo sobre el cuento**.
- **Balanceo por temática** + limpieza de títulos basura (arregla el colapso a plantilla genérica de mT5).
- **Checkpoints a Drive** con **reanudación** (Trainer) + split de validación.

> Stack: `transformers` + `peft` + `Trainer` (sin TRL, para evitar roturas por versión).


## Paso 0 — Setup (GPU, dependencias, Drive)

In [ ]:
import os, sys
EN_COLAB = "google.colab" in sys.modules
print("En Colab:", EN_COLAB)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "Sin GPU: Runtime > GPU (A100)"

if EN_COLAB:
    # transformers>=4.51 es necesario para Qwen3. -U trae lo ultimo.
    !pip install -q -U transformers peft accelerate datasets bitsandbytes
    # Colab trae un torchao viejo (0.10) que rompe a peft (exige >0.16). No lo usamos -> lo quitamos.
    !pip uninstall -q -y torchao
    from google.colab import drive
    drive.mount("/content/drive")

# Llama-3.1 es GATED: 1) pide acceso en su pagina de HF, 2) crea un token (Settings > Access Tokens),
# 3) define HF_TOKEN en el entorno o en los Secrets de Colab. Para modelos Qwen esto NO hace falta.
# IMPORTANTE: no pegues el token en el codigo; usa una variable de entorno / Secret.
from huggingface_hub import login
_hf_token = os.environ.get("HF_TOKEN", "")
if _hf_token:
    login(_hf_token)
else:
    print("HF_TOKEN no definido: ok para modelos Qwen; necesario para Llama-3.1 (gated).")


## Paso 1 — Configuración

Guía por GPU (bf16 LoRA): el A100 entra de sobra con Qwen3-4B. Si usas Llama-3.1-8B, activa `USAR_QLORA=True`.


In [11]:
import os, torch

# === Modelo ===
MODELO_BASE = "meta-llama/Llama-3.1-8B-Instruct"   # GATED: requiere token (ver Paso 0)
USAR_QLORA  = True      # 8B en A100 40GB necesita 4-bit. (Alternativa sin token: "Qwen/Qwen3-4B-Instruct-2507" con USAR_QLORA=False)

# === Datos ===
RUTA_DATASET = "/content/drive/MyDrive/dataset_cuentos_ampliado.csv"  # ajusta a tu ruta
USAR_TITULO  = True     # mete el titulo en el prompt (se limpian los titulos basura)
BALANCEAR    = True     # tope por tematica para evitar el colapso a la plantilla generica
TOPE_POR_TEMA = 500     # maximo de cuentos por tematica si BALANCEAR=True
VAL_FRAC     = 0.05
SEED         = 42

# === Entrenamiento (LoRA) ===
MAX_LEN     = 1280      # tokens totales (prompt + cuento). Lo que sobre se trunca
LOTE        = 4
ACUMULAR    = 4         # lote efectivo = 16
EPOCAS      = 3
LR          = 2e-4      # tipico para LoRA
LORA_R      = 16
LORA_ALPHA  = 32
LORA_DROPOUT= 0.05
GRAD_CKPT   = True
GUARDAR_CADA_PASOS = 100   # checkpoints a Drive (crash-safe) + evaluacion

# === Salida (Drive) ===
# Cambia esto:
RUTA_SALIDA = "/content/drive/MyDrive/cuentos_modelo/llama_lora"
RUTA_MEJOR  = os.path.join(RUTA_SALIDA, "mejor")
os.makedirs(RUTA_SALIDA, exist_ok=True)


SYSTEM = ("Eres un escritor de cuentos infantiles en español. Escribe cuentos originales, "
          "claros, con inicio, desarrollo y final, apropiados para ninos.")

import random, numpy as np
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("Modelo:", MODELO_BASE, "| QLoRA:", USAR_QLORA)
print("Dataset:", RUTA_DATASET, "| existe:", os.path.exists(RUTA_DATASET))


Modelo: meta-llama/Llama-3.1-8B-Instruct | QLoRA: True
Dataset: /content/drive/MyDrive/dataset_cuentos_ampliado.csv | existe: True


## Paso 2 — Cargar, limpiar títulos y balancear por temática

In [12]:
import csv, re
from collections import Counter, defaultdict
csv.field_size_limit(50*1024*1024)

def titulo_basura(t):
    """Detecta titulos que en realidad son fechas/numeros/ruido (vienen de datos externos)."""
    t = (t or "").strip()
    if len(t) < 3: return True
    if re.fullmatch(r"[\d\s\-–—.]+", t): return True          # solo numeros/fechas: "1833-1883"
    if re.fullmatch(r"\d{3,4}\s*[-–]\s*\d{3,4}", t): return True
    return False

filas = []
with open(RUTA_DATASET, encoding="utf-8-sig") as f:

    for fila in csv.DictReader(f):
        texto = (fila["texto"] or "").strip()
        tema  = (fila["tematica"] or "").strip()
        if not texto or not tema: continue
        tit = (fila.get("titulo") or "").strip()
        if titulo_basura(tit): tit = ""     # se cae a prompt solo-tematica
        filas.append({"tematica": tema, "titulo": tit, "texto": texto})

print("Cuentos cargados:", len(filas))
print("Con titulo util:", sum(1 for x in filas if x['titulo']), "| sin titulo (basura o vacio):",
      sum(1 for x in filas if not x['titulo']))

# Balanceo: tope por tematica
if BALANCEAR:
    por_tema = defaultdict(list)
    for x in filas: por_tema[x["tematica"]].append(x)
    rng = random.Random(SEED)
    balanceado = []
    for t, xs in por_tema.items():
        rng.shuffle(xs)
        balanceado.extend(xs[:TOPE_POR_TEMA])
    filas = balanceado
    print("\nTras balanceo (tope", TOPE_POR_TEMA, "por tema):", len(filas))

print("\nDistribucion por tematica:")
for t, n in Counter(x["tematica"] for x in filas).most_common():
    print(f"  {t:28s} {n}")


Cuentos cargados: 9477
Con titulo util: 9475 | sin titulo (basura o vacio): 2

Tras balanceo (tope 500 por tema): 4537

Distribucion por tematica:
  robots_y_tecnologia          500
  dinosaurios_y_prehistoria    500
  princesas_y_castillos        500
  piratas                      500
  naturaleza_y_bosques         500
  fabulas                      500
  magia_y_brujas               500
  animales                     385
  fantasmas_y_misterio         336
  mar_y_oceano                 138
  mitos_leyendas               113
  heroes_y_aventuras           38
  espacio                      20
  monstruos_y_criaturas        7


## Paso 3 — Formato chat + máscara + split estratificado

Cada ejemplo se convierte a chat (system/user/assistant). La pérdida se calcula **solo sobre el cuento**
(el prompt se enmascara con -100), que es lo correcto para SFT.


In [16]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset as TorchDataset

tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def construir_user(tematica, titulo):
    tema = tematica.replace("_", " ").strip()
    if USAR_TITULO and titulo:
        return f'Escribe un cuento infantil titulado "{titulo}" sobre {tema}.'
    return f"Escribe un cuento infantil sobre {tema}."

def formatear(ejemplo):
    user = construir_user(ejemplo["tematica"], ejemplo["titulo"])
    prompt_msgs = [{"role": "system", "content": SYSTEM},
                   {"role": "user", "content": user}]
    prompt_text = tokenizer.apply_chat_template(
        prompt_msgs, tokenize=False, add_generation_prompt=True)
    full_msgs = prompt_msgs + [{"role": "assistant", "content": ejemplo["texto"]}]
    full_text = tokenizer.apply_chat_template(
        full_msgs, tokenize=False, add_generation_prompt=False)

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids   = tokenizer(full_text,   add_special_tokens=False)["input_ids"]
    full_ids   = full_ids[:MAX_LEN]

    labels = list(full_ids)
    n_prompt = min(len(prompt_ids), len(full_ids))
    for i in range(n_prompt):
        labels[i] = -100

    return {
        "input_ids":      list(full_ids),
        "labels":         labels,
        "attention_mask": [1] * len(full_ids)
    }

rng = random.Random(SEED)
idx_por_tema = defaultdict(list)
for i, x in enumerate(filas): idx_por_tema[x["tematica"]].append(i)
idx_val = set()
for t, idxs in idx_por_tema.items():
    rng.shuffle(idxs); idx_val.update(idxs[:max(1, int(len(idxs)*VAL_FRAC))])

train = [formatear(filas[i]) for i in range(len(filas)) if i not in idx_val]
val   = [formatear(filas[i]) for i in range(len(filas)) if i in idx_val]

class CuentosTokenizado(TorchDataset):
    def __init__(self, ejemplos): self.ejemplos = ejemplos
    def __len__(self): return len(self.ejemplos)
    def __getitem__(self, i): return self.ejemplos[i]

ds_train = CuentosTokenizado(train)
ds_val   = CuentosTokenizado(val)
print("Train:", len(ds_train), "| Val:", len(ds_val))

Train: 4313 | Val: 224


## Paso 4 — Modelo base + LoRA

In [17]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if USAR_QLORA:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    modelo = AutoModelForCausalLM.from_pretrained(MODELO_BASE, quantization_config=bnb,
                                                  torch_dtype=torch.bfloat16, device_map={"": 0})
    modelo = prepare_model_for_kbit_training(modelo, use_gradient_checkpointing=GRAD_CKPT)
else:
    modelo = AutoModelForCausalLM.from_pretrained(MODELO_BASE, torch_dtype=torch.bfloat16).to("cuda")

modelo.config.use_cache = False
lora = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
                  task_type="CAUSAL_LM",
                  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
modelo = get_peft_model(modelo, lora)
if GRAD_CKPT:
    modelo.gradient_checkpointing_enable(); modelo.enable_input_require_grads()
modelo.print_trainable_parameters()


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


## Paso 5 — Entrenamiento (checkpoints a Drive + reanudación)

`Trainer` guarda a Drive cada `GUARDAR_CADA_PASOS`, evalúa, conserva el mejor y se puede **reanudar**:
si la sesión se cae, vuelve a correr desde el Paso 0 y detecta el último checkpoint automáticamente.


In [18]:
import glob
from dataclasses import dataclass
from transformers import Trainer, TrainingArguments

@dataclass
class ColadorCausal:
    pad_id: int
    def __call__(self, feats):
        maxlen = max(len(f["input_ids"]) for f in feats)
        ii, ll, am = [], [], []
        for f in feats:
            d = maxlen - len(f["input_ids"])
            ii.append(f["input_ids"] + [self.pad_id]*d)
            ll.append(f["labels"]    + [-100]*d)
            am.append(f["attention_mask"] + [0]*d)
        return {"input_ids": torch.tensor(ii), "labels": torch.tensor(ll),
                "attention_mask": torch.tensor(am)}

args = TrainingArguments(
    output_dir=RUTA_SALIDA,
    per_device_train_batch_size=LOTE,
    per_device_eval_batch_size=LOTE,
    gradient_accumulation_steps=ACUMULAR,
    num_train_epochs=EPOCAS,
    learning_rate=LR,
    warmup_ratio=0.06,
    lr_scheduler_type="cosine",
    logging_steps=20,
    eval_strategy="steps", eval_steps=GUARDAR_CADA_PASOS,
    save_strategy="steps", save_steps=GUARDAR_CADA_PASOS,
    save_total_limit=3,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    bf16=True, gradient_checkpointing=GRAD_CKPT,
    report_to="none", remove_unused_columns=False,
)

trainer = Trainer(model=modelo, args=args, train_dataset=ds_train, eval_dataset=ds_val,
                  data_collator=ColadorCausal(tokenizer.pad_token_id))

hay_ckpt = bool(glob.glob(os.path.join(RUTA_SALIDA, "checkpoint-*")))
print("Reanudando desde checkpoint:" , hay_ckpt)
trainer.train(resume_from_checkpoint=hay_ckpt)
print("Mejor eval_loss:", trainer.state.best_metric)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Reanudando desde checkpoint: False


Step,Training Loss,Validation Loss
100,1.818131,1.821303
200,1.782483,1.777298
300,1.627648,1.769223
400,1.640102,1.750294
500,1.575809,1.732334
600,1.438471,1.760956
700,1.411814,1.762375
800,1.396688,1.758874
810,1.396688,1.758873


Mejor eval_loss: 1.732334017753601


## Paso 6 — Guardar el adapter LoRA (el mejor)

In [19]:
trainer.save_model(RUTA_MEJOR)      # guarda solo el adapter LoRA (ligero)
tokenizer.save_pretrained(RUTA_MEJOR)
print("Adapter guardado en:", RUTA_MEJOR)

# (Opcional) fusionar LoRA en el modelo base para inferencia mas simple/rapida:
# fusionado = modelo.merge_and_unload()
# fusionado.save_pretrained(RUTA_SALIDA + "/fusionado"); tokenizer.save_pretrained(RUTA_SALIDA + "/fusionado")


Adapter guardado en: /content/drive/MyDrive/cuentos_modelo/llama_lora/mejor


## Paso 7 — Generar cuentos

In [20]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(RUTA_MEJOR)
base = AutoModelForCausalLM.from_pretrained(MODELO_BASE, torch_dtype=torch.bfloat16).to("cuda")
mod = PeftModel.from_pretrained(base, RUTA_MEJOR).eval()

def construir_user(tematica, titulo=None):
    tema = tematica.replace("_", " ").strip()
    if titulo: return f'Escribe un cuento infantil titulado "{titulo}" sobre {tema}.'
    return f"Escribe un cuento infantil sobre {tema}."

@torch.no_grad()
def generar(tematica, titulo=None, temperatura=0.7, max_new=700):
    msgs = [{"role":"system","content":SYSTEM},
            {"role":"user","content":construir_user(tematica, titulo)}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors="pt").to(mod.device)
    out = mod.generate(**inp, max_new_tokens=max_new, do_sample=True,
                       temperature=temperatura, top_p=0.8, top_k=20, repetition_penalty=1.1)
    return tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)

# Recuerda usar el slug EXACTO de la tematica (p.ej. princesas_y_castillos, no "castillos")
for tema in ["piratas", "princesas_y_castillos", "robots_y_tecnologia", "naturaleza_y_bosques"]:
    print(f"================ {tema} ================")
    print(generar(tema), "\n")


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


================ piratas ================


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Un día, una mujer estaba lavando su ropa en el río cuando vio a un hombre hundirse en el agua.
La mujer se acercó al lugar donde había visto al hombre y lo sacó del agua.
Pero no era un hombre sino un pescado. Era enorme y estaba cubierto de lodo y arena.
La mujer lo dejó en la orilla y volvió a casa.
A los pocos días, el marido regresó del trabajo y encontró al pescado en la casa.
—¿De dónde lo has sacado? —preguntó.
—Lo vi en el río mientras lavaba la ropa y lo saqué del agua.
El marido dijo que sería buena idea cocinarlo y así lo hicieron.
Mientras comían, el pescado les preguntó:
—¿Por qué me mataron?
—Porque eres un pescado y como todos los peces, debemos cocinarte.
—No soy ningún pescado —dijo el pescado—. Soy un príncipe. Un hechizo me convirtió en pescado y ahora estoy libre gracias a ti. Tengo un reino lejos de aquí y quiero llevarte contigo.
Y se fueron juntos.
Pasaron varios años y tuvieron dos hijos.
Cuando llegaron al reino, el príncipe fue recibido por todos como un héroe

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


La Princesa Ileana vivía en el Reino del Canto, donde la música era el único lenguaje permitido dentro de las murallas reales. Su padre, el Rey Melodius, exigía que cada habitante aprendiera a tocar al menos una melodía antes de cumplir los diez años. La princesa, sin embargo, sentía que su corazón cantaba en silencio, y sus manos temblaban cuando intentaba tocar las notas que le enseñaban.

Un día, mientras exploraba el gran bosque que rodeaba el castillo, descubrió un arroyo misterioso cuyas aguas parecían cambiar de tono según la hora del día. Fascinada, comenzó a observar cómo la luz reflejaba en ellas diferentes colores. Aprendió a reconocer patrones y a comprender la naturaleza del sonido.

Cuando regresó al palacio, no podía seguir fingiendo ser como todos los demás. Le dijo a su madre que prefería observar el mundo que aprender canciones. El rey se enfadó tanto que ordenó que la encerraran en una torre de silencio, esperando que el tiempo la convirtiera en una artista como toda

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


La vida es una aventura que se vive al revés. En el fondo del pozo de los sueños, donde la tierra es negra como la noche y las paredes están cubiertas de musgo húmedo, hay un árbol de sombras cuyas raíces llegan hasta la luna. Las ramas están llenas de hojas verdes que brillan con un resplandor extraño. Pero no hay fruto en ellas. No hay flores, ni pájaros, ni insectos. Solo hojas que parecen vivir sin esperanza.

En uno de sus troncos había un agujero pequeño por el que asomaba una luz azul. Un día, un joven llamado Aroto se deslizó por él. Se encontró en un mundo subterráneo, donde todo era diferente. Había plantas que crecían hacia arriba desde el suelo, y animales que corrían con patas traseras. Todo estaba iluminado por una luz azulada que venía de una fuente misteriosa.

Aroto siguió el río que fluye hacia el centro de la Tierra. La corriente lo llevó a una gran sala donde se celebraban consejos entre los habitantes del subsuelo. Allí escuchó historias antiguas de una época en qu

In [21]:
print(generar("un cuento sobre dos novios perdidos en el bosque"))

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Un día, una niña se quedó sola al lado del río mientras su hermano mayor iba a buscar agua. De pronto oyó pasos que venían de lejos y, como tenía miedo, corrió hacia donde estaba su hermano. Pero cuando llegó allí ya no había nadie.
Entonces la pequeña comenzó a llorar desconsoladamente y a gritar:
-¡Ay! ¡Mi hermanito ha desaparecido!
El río era muy ancho y profundo; por eso los vecinos no podían ayudarla. Entonces ella siguió llamando a su hermano y él respondió desde el otro lado:
-¡No llores, hermanita! Te ayudaré a pasar.
Y dijo que le daría una mano para cruzarlo. La niña obedeció y fue pasando de una orilla a otra. Cuando estuvo cerca de él, este se lanzó contra ella, la tomó por los hombros y tiró de ella hacia sí. La muchachita, asustada, le dijo:
-No te voy a dejar nunca más sola, porque te quiero mucho.
FIN
